# 🎭 Nexa — Diffusion-Based Face Swapper

**IP-Adapter FaceID + Stable Diffusion 1.5** face-swap pipeline.

### Requirements
- Google Colab with **GPU runtime** (T4 recommended)
- Upload your `source.jpg` (face to use) and `target.jpg` (image to swap into) to `/content/`

> **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
#@title 1️⃣ Clone Repository & Install Dependencies
import os

# Clone the repo (change URL to your fork if needed)
REPO_URL = "https://github.com/YOUR_USERNAME/nexa.git"  # ← UPDATE THIS
REPO_DIR = "/content/nexa"

if not os.path.exists(REPO_DIR):
    # If you uploaded the nexa folder directly, skip cloning
    print("⚠️  Repository not found. You can either:")
    print("    1. Upload the 'nexa' folder to /content/")
    print("    2. Update REPO_URL above and uncomment the git clone line")
    print()
    # !git clone $REPO_URL $REPO_DIR
else:
    print(f"✅ Repository found at {REPO_DIR}")

# Alternative: Upload as zip and extract
# from google.colab import files
# uploaded = files.upload()  # upload nexa.zip
# !unzip -o nexa.zip -d /content/

In [ ]:
#@title 2️⃣ Install Nexa Package
import os
os.chdir("/content/nexa")

# Uninstall conflicting onnxruntime before installing GPU version
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null

# Install nexa with GPU extras
!pip install -e ".[gpu]" 2>&1 | tail -5

# Verify installation
!python -c "import nexa; print(f'Nexa v{nexa.__version__} installed successfully')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')"

In [ ]:
#@title 3️⃣ Install FFmpeg (for video processing)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1
print("✅ FFmpeg ready")

In [ ]:
#@title 4️⃣ Upload Source & Target Images
import os
from google.colab import files
from IPython.display import display, Image as IPImage

# Option A: Upload via Colab file picker
print("Upload your SOURCE face image (the face identity to use):")
uploaded_source = files.upload()
source_name = list(uploaded_source.keys())[0]
source_path = f"/content/{source_name}"
with open(source_path, 'wb') as f:
    f.write(uploaded_source[source_name])
print(f"✅ Source saved: {source_path}")
display(IPImage(source_path, width=256))

print("\nUpload your TARGET image (the image where faces will be swapped):")
uploaded_target = files.upload()
target_name = list(uploaded_target.keys())[0]
target_path = f"/content/{target_name}"
with open(target_path, 'wb') as f:
    f.write(uploaded_target[target_name])
print(f"✅ Target saved: {target_path}")
display(IPImage(target_path, width=256))

In [ ]:
#@title 5️⃣ Run Face Swap (Image Mode)
#@markdown ### Configuration
steps = 20  #@param {type:"slider", min:10, max:40, step:5}
strength = 0.65  #@param {type:"slider", min:0.3, max:0.9, step:0.05}
guidance_scale = 5.0  #@param {type:"slider", min:2.0, max:10.0, step:0.5}
ip_scale = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
enhancer = "none"  #@param ["none", "gfpgan"]

output_path = "/content/output.jpg"

# Build the command
cmd = f"""nexa \\
  --source {source_path} \\
  --target {target_path} \\
  --output {output_path} \\
  --steps {steps} \\
  --strength {strength} \\
  --guidance-scale {guidance_scale} \\
  --ip-scale {ip_scale} \\
  --gpu"""

if enhancer != "none":
    cmd += f" --enhancer {enhancer}"

print(f"Running:\n{cmd}\n")
!{cmd}

# Display result
from IPython.display import display, Image as IPImage
import os
if os.path.exists(output_path):
    print("\n✅ Face swap complete!")
    display(IPImage(output_path, width=512))
else:
    print("❌ Output not found — check errors above.")

In [ ]:
#@title 6️⃣ Alternative: Run via Python API (more control)
import cv2
import numpy as np
from IPython.display import display, Image as IPImage
import tempfile

# Import Nexa modules
from nexa.core.pipeline import NexaPipeline

# Initialize pipeline
pipeline = NexaPipeline(
    model_id="runwayml/stable-diffusion-v1-5",
    device="cuda",
    steps=20,
    enhancer_name=None,  # Set to "gfpgan" for enhancement
    threshold=0.6,
    ip_scale=1.0,
    strength=0.65,
    guidance_scale=5.0,
)

# Run swap
result_path = pipeline.process_image_single(
    source_path=source_path,
    target_path=target_path,
    output_path="/content/output_api.jpg",
)

print(f"✅ Result saved to: {result_path}")
display(IPImage(str(result_path), width=512))

In [ ]:
#@title 7️⃣ Video Face Swap (Optional)
#@markdown Upload a video file and run face swap on all frames.

from google.colab import files

print("Upload your TARGET video:")
uploaded_video = files.upload()
video_name = list(uploaded_video.keys())[0]
video_path = f"/content/{video_name}"
with open(video_path, 'wb') as f:
    f.write(uploaded_video[video_name])
print(f"✅ Video saved: {video_path}")

video_output = "/content/output_video.mp4"

!nexa \\
  --source {source_path} \\
  --target {video_path} \\
  --output {video_output} \\
  --steps 15 \\
  --gpu

import os
if os.path.exists(video_output):
    print(f"\n✅ Video saved: {video_output}")
    # Download the result
    files.download(video_output)
else:
    print("❌ Video output not found.")

In [ ]:
#@title 8️⃣ Download Results
from google.colab import files
import os

for f in ["/content/output.jpg", "/content/output_api.jpg", "/content/output_video.mp4"]:
    if os.path.exists(f):
        print(f"Downloading: {f}")
        files.download(f)

print("\n✅ Done!")

In [ ]:
#@title 9️⃣ Side-by-Side Comparison
import cv2
import numpy as np
from IPython.display import display, Image as IPImage
import os

output_file = "/content/output.jpg"
if not os.path.exists(output_file):
    output_file = "/content/output_api.jpg"

if os.path.exists(target_path) and os.path.exists(output_file):
    target_img = cv2.imread(target_path)
    output_img = cv2.imread(output_file)

    # Resize to same height
    h = min(target_img.shape[0], output_img.shape[0], 512)
    target_img = cv2.resize(target_img, (int(target_img.shape[1] * h / target_img.shape[0]), h))
    output_img = cv2.resize(output_img, (int(output_img.shape[1] * h / output_img.shape[0]), h))

    # Add labels
    cv2.putText(target_img, 'ORIGINAL', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(output_img, 'SWAPPED', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Concatenate side by side
    comparison = np.hstack([target_img, output_img])
    comp_path = "/content/comparison.jpg"
    cv2.imwrite(comp_path, comparison)
    display(IPImage(comp_path, width=1024))
else:
    print("Run a face swap first!")